# Turn SEC EDGAR into a knowledge graph Claude can query

Investor research over SEC filings is a SQL classic — companies, filings, insider transactions, institutional holdings, board members, all in tables. What's different here:

1. **The same person is one node, no matter how many roles or companies.** Tim Cook is officer + director at Apple in a single `:Person`; Warren Buffett's the trifecta (officer + director + 10%-owner of Berkshire); John D Rainey is CFO at Walmart AND director at Microsoft. Pre-unification all of these would have been 2–3 disconnected nodes. Multi-role / cross-company queries become a single MATCH instead of three SQL joins.
2. **Sector, fund, board** all materialise as graph structure. SIC code is a `:SicCode` node (not a string column). 13F filers that are also operating companies (BlackRock, Berkshire, JPM, GS) link via `:IS_COMPANY` to their `:Company` self-node. Board directors fan out from one `:Person` to every company they sit on.

Below: ~3 minutes of build (10 tickers across sectors), then 7 Cypher queries that have no clean SQL equivalent. The final cell registers the graph as a Claude Desktop MCP server.

Requirements: `pip install kglite` plus the Claude Desktop app for the MCP step. First build ~2-3 min (SEC's 10 req/s rate limit + ~145 MB submissions.zip). Subsequent builds with the same scope are cache hits.

In [ ]:
from pathlib import Path

from kglite.datasets.sec import SEC

# SEC fair-access policy requires a descriptive User-Agent (your
# name + contact email). Generic UAs get 403'd.
USER_AGENT = "Your Name your@email.com"

# 10 tickers across sectors so cross-company patterns (board
# interlocks, sector-cohort sells, fund-as-issuer) actually
# materialise. 0.9.45+: pass tickers OR int CIKs (case-insensitive
# lookup via SEC's company_tickers.json, cached after first fetch).
graph = SEC.open(
    Path.home() / ".kglite-sec-demo",
    cik_list=[
        "AAPL",
        "MSFT",
        "GOOGL",
        "NVDA",  # Big tech
        "BRK-B",
        "JPM",
        "GS",  # Finance (BRK is also a 13F filer)
        "TSLA",  # Auto
        "WMT",
        "HD",  # Retail
    ],
    years=1,  # 1 year of shallow Filing index
    detailed=1,  # 1 year of per-filing payloads
    # (Form 4 / 13F / 8-K / SC 13D / DEF 14A / Exhibit 21)
    mode="memory",  # auto-escalates to mapped/disk for bigger slices
    user_agent=USER_AGENT,
)
print(f"\n{graph}\n")

# Cypher: MATCH (var:NodeType)-[:EDGE_TYPE]->(other) ... RETURN ...
# Full reference: https://kglite.readthedocs.io (CYPHER.md)
print("Companies in scope:")
for r in graph.cypher(
    "MATCH (c:Company)-[:IN_INDUSTRY]->(s:SicCode) "
    "RETURN c.name AS company, c.tickers AS tickers, s.description AS sector "
    "ORDER BY company"
):
    print(f"  {r['company']:25} {r['tickers']:18} {r['sector']}")

## Seven queries the graph wins on

Each below uses pattern matching across typed edges that would be painful (and slow) in SQL — the joins multiply and the planner can't index by edge type.

In [ ]:
# Q1 — Multi-role insiders at the same company. Tim Cook is both an
# officer AND director at Apple; Warren Buffett's the trifecta
# (officer + director + 10% owner) at Berkshire. Pre-J3 these would
# have been disconnected nodes (Person vs Director); now they
# collapse onto one :Person with multiple typed edges back to the
# company.
print("Q1: Multi-role insiders (the unified-Person payoff)")
for r in graph.cypher("""
    MATCH (p:Person)<-[r]-(c:Company)
    WHERE type(r) IN ['IS_OFFICER_OF','IS_DIRECTOR_OF','IS_BENEFICIAL_OWNER_OF']
    WITH p, c, collect(DISTINCT type(r)) AS roles
    WHERE size(roles) >= 2
    RETURN p.display_name AS person, c.name AS company, roles
    ORDER BY size(roles) DESC, person LIMIT 10
"""):
    print(f"  {r['person']:25}  {r['company']:25}  {r['roles']}")

# Q2 — Cross-company board overlap. Persons who appear on multiple
# companies' insider rolls (officer at one, director at another, or
# director at both). On a 3-CIK demo this often returns 0 rows
# because in-scope boards don't share members; widen `cik_list`
# (e.g. 10 major filers across sectors) to see board interlocks
# light up.
print("\nQ2: Cross-company board members")
for r in graph.cypher("""
    MATCH (c1:Company)-[r1]->(p:Person)<-[r2]-(c2:Company)
    WHERE c1.cik < c2.cik
      AND type(r1) IN ['IS_OFFICER_OF','IS_DIRECTOR_OF']
      AND type(r2) IN ['IS_OFFICER_OF','IS_DIRECTOR_OF']
    RETURN p.display_name AS person, c1.name AS at1, c2.name AS at2,
           type(r1) AS r1, type(r2) AS r2
    LIMIT 10
"""):
    print(f"  {r['person']:25}  {r['r1']}@{r['at1']:18}  {r['r2']}@{r['at2']}")

In [ ]:
# Q3 — Sector cohort insider sells. SicCode is a node, so we walk
# through the industry to find every officer-sell across that
# sector — no `GROUP BY sic` ceremony.
print("Q3: Insider sells aggregated by sector")
for r in graph.cypher("""
    MATCH (sic:SicCode)<-[:IN_INDUSTRY]-(c:Company)-[:IS_OFFICER_OF]->(p:Person)
    MATCH (t:Transaction {transaction_code: 'S'})-[:OF_PERSON]->(p)
    MATCH (t)-[:INVOLVES_ISSUER]->(c)
    RETURN sic.description AS sector, c.name AS company,
           p.display_name AS officer, sum(t.shares) AS shares_sold
    ORDER BY shares_sold DESC LIMIT 10
"""):
    print(f"  {r['sector']:28} {r['company']:18} {r['officer']:22} {r['shares_sold']:>12,.0f} sh")

# Q4 — Fund-as-issuer. Berkshire is both a 13F filer AND an
# operating company; the :IS_COMPANY edge bridges the two roles
# in one MATCH.
print("\nQ4: 13F filers that are also operating companies in this scope")
for r in graph.cypher("""
    MATCH (m:InstitutionalManager)-[:IS_COMPANY]->(c:Company)
    OPTIONAL MATCH (c)<-[:FILED_BY]-(f:Filing)
    RETURN c.name AS company, count(DISTINCT f) AS filings_as_issuer
    ORDER BY filings_as_issuer DESC
"""):
    print(f"  {r['company']:25}  {r['filings_as_issuer']} filings as issuer")

In [ ]:
# Q5 — 8-K → Form 4 proximity. Detect officer trades that follow
# Item 5.02 (officer-change) events. A natural 4-hop pattern; the
# SQL equivalent needs window joins over date arithmetic.
print("Q5: Form 4 trades by officers after 8-K Item 5.02 events")
for r in graph.cypher("""
    MATCH (e:Event {item_code: '5.02'})-[:OF_FILING]->(f1:Filing)-[:FILED_BY]->(c:Company)
    MATCH (c)-[:IS_OFFICER_OF]->(p:Person)
    MATCH (t:Transaction)-[:OF_PERSON]->(p)
    MATCH (t)-[:INVOLVES_ISSUER]->(c)
    MATCH (t)-[:REPORTED_IN_FILING]->(f2:Filing)
    WHERE f2.filed_date > f1.filed_date
    RETURN c.name AS company, p.display_name AS officer,
           e.description AS event, t.transaction_date AS tx_date,
           t.shares AS shares
    LIMIT 5
"""):
    print(f"  {r['company']:18} {r['officer']:22} {r['tx_date']} {r['shares']:>10,.0f} sh — {r['event'][:60]}")

# Q6 — Concentrated voting power. 13F voting tiers were always
# captured; with edge properties the pattern is direct.
print("\nQ6: Concentrated voting positions (≥1M sole-voted shares)")
for r in graph.cypher("""
    MATCH (m:InstitutionalManager)-[h:HOLDS]->(s:Security)
    WHERE h.voting_sole >= 1000000
    RETURN m.name AS firm, s.name AS security, h.voting_sole AS sole_votes
    ORDER BY sole_votes DESC LIMIT 5
"""):
    print(f"  {r['firm']:30} {r['security']:25} {r['sole_votes']:>12,.0f}")

# Q7 — Subsidiary depth. Parent → child traversal via Exhibit 21.
print("\nQ7: Subsidiary counts (from Exhibit 21 of 10-K filings)")
for r in graph.cypher("""
    MATCH (c:Company)<-[:OF_COMPANY]-(s:Subsidiary)
    WITH c, count(s) AS subs
    RETURN c.name AS company, subs ORDER BY subs DESC
"""):
    print(f"  {r['company']:25}  {r['subs']:>4} subsidiaries")

In [ ]:
# graph.describe() emits a compact XML schema — every node type +
# its properties, every edge + direction, exploration hints — that
# fits in a system prompt. It's how the MCP server tells Claude
# what's in the graph before it writes its first cypher_query().
xml = graph.describe()
print(xml[:700])
print(f"\n... [truncated — full schema is {len(xml):,} chars / {xml.count(chr(10))} lines]")

## Hand the same capability to Claude Desktop

After the next cell you can ask Claude things like *"who's been selling AAPL insider lately?"* — the agent reads the schema with `graph_overview()`, writes Cypher against `:Person → :Transaction → :Company`, and answers from real Form 4 data.

This uses the `--graph X.kgl` MCP pattern (one curated graph), contrasting with the `workspace` clone-on-demand pattern in [`examples/codebase_to_claude_mcp.ipynb`](https://github.com/kkollsga/kglite/blob/main/examples/codebase_to_claude_mcp.ipynb).

In [ ]:
from pathlib import Path

from kglite.mcp_server import claude_config

DRY_RUN = True  # flip to False to write the Claude Desktop config

kgl_path = Path.home() / ".kglite-sec-demo" / "sec.kgl"
graph.save(str(kgl_path))
print(f"Saved graph: {kgl_path}")

# Tiny manifest. `save_graph: true` lets Claude persist Cypher
# SET/CREATE annotations (opt-in since 0.9.41).
manifest_path = kgl_path.parent / "sec_mcp.yaml"
manifest_path.write_text("""\
name: SEC Filings
builtins:
  save_graph: true
instructions: |
  SEC EDGAR knowledge graph (AAPL, BRK-B, TSLA, 1 year detailed).
  Use cypher_query + graph_overview to answer investor questions.
  Person nodes subsume both Form 4 reporters and DEF 14A directors,
  linked to Company via IS_OFFICER_OF / IS_DIRECTOR_OF /
  IS_BENEFICIAL_OWNER_OF / SERVES_ON_BOARD typed edges.
  Company.tickers carries the ticker (e.g. 'AAPL') as a property.
""")

entry = claude_config.add_mcp(
    name="sec-filings",
    command="kglite-mcp-server",
    args=["--graph", str(kgl_path), "--mcp-config", str(manifest_path)],
    client="claude_desktop",
    overwrite=True,
    dry_run=DRY_RUN,
)

cfg_path = claude_config.default_path("claude_desktop")
print(f"{'Would write' if DRY_RUN else 'Wrote'} entry to {cfg_path}:")
print(entry)
if not DRY_RUN:
    print("\nRestart Claude Desktop, then ask:")
    print("  -> Who's been selling AAPL insider in the past year?")

## What you have now

After restarting Claude Desktop, the **sec-filings** MCP server shows up in Claude's tools. Useful prompts:

- *"Show me Tim Cook's Form 4 sales in the past year, ordered by date and dollar value."*
- *"Which Berkshire 13F positions are over $1B sole-voted?"*
- *"Find anyone who serves on multiple boards in this graph."*
- *"For each company, count 8-K filings by Item code. Highlight Item 5.02 spikes."*

Under the hood Claude calls `graph_overview()` (reads the XML schema), then `cypher_query(...)` (against the graph the previous cell built), and answers in natural language with real rows.

**Scaling up:**

- Drop the `cik_list` to ingest the full universe (~6,000 filers). The build escalates `mode` automatically: predicted <4 GB stays in `memory`, 4–16 GB → `mapped`, >16 GB → `disk`. Full universe + 1 year detailed is ~80 minutes at SEC's 10 req/s rate limit; subsequent runs hit the cache.
- Bump `years` (or `years="all"` for 1993→present) and `detailed` for deeper historical coverage.
- Re-running `SEC.open` with the same args is a cache hit — `raw/` is immutable, `processed/` regenerates only when `raw/` changes, `graph/{mode}/` is loaded directly.

**Next steps:**

- Repo: [github.com/kkollsga/kglite](https://github.com/kkollsga/kglite)
- Docs: [kglite.readthedocs.io](https://kglite.readthedocs.io)
- SEC loader guide: [docs/guides/sec.md](https://kglite.readthedocs.io/en/latest/guides/sec.html) — full schema (11 node types + 16 edges), slice grammar, storage trajectory.
- Companion notebook (code-graph pattern): [`examples/codebase_to_claude_mcp.ipynb`](https://github.com/kkollsga/kglite/blob/main/examples/codebase_to_claude_mcp.ipynb).